In [1]:
import keras
from keras import layers
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [10]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

# Custom F1 metric
class F1Score(keras.metrics.Metric):
    def __init__(self, name="f1_score", **kwargs):
        super().__init__(name=name, **kwargs)
        self.precision = keras.metrics.Precision()
        self.recall = keras.metrics.Recall()

    def update_state(self, y_true, y_pred, sample_weight=None):
        self.precision.update_state(y_true, y_pred, sample_weight)
        self.recall.update_state(y_true, y_pred, sample_weight)

    def result(self):
        p = self.precision.result()
        r = self.recall.result()
        return 2 * ((p * r) / (p + r + tf.keras.backend.epsilon()))

    def reset_state(self):
        self.precision.reset_state()
        self.recall.reset_state()

def plot_metrics(history):
    # Plot Accuracy
    plt.plot(history.history['accuracy'], label='train acc')
    plt.plot(history.history['val_accuracy'], label='val acc')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()

    # Plot AUC
    plt.plot(history.history['auc'], label='train AUC')
    plt.plot(history.history['val_auc'], label='val AUC')
    plt.title('Model AUC')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    plt.legend()
    plt.show()

    # Plot F1
    plt.plot(history.history['f1'], label='train F1')
    plt.plot(history.history['val_f1'], label='val F1')
    plt.title('Model F1 Score')
    plt.xlabel('Epoch')
    plt.ylabel('F1')
    plt.legend()
plt.show()

In [2]:
data = np.load("OTIDS_clean_data.npz")
# Access arrays by the names you used when saving
X_train = data["X_train"]
X_test  = data["X_test"]
y_train = data["y_train"]
y_test  = data["y_test"]
y_train = y_train.astype("int32").ravel()
y_test  = y_test.astype("int32").ravel()

In [15]:

#1d cnn sequential model:
def create_model_otids(): #this is the same model we'll always use for all.
    model = keras.Sequential()
    model.add(layers.Input(shape=(600, 11)))
    model.add(layers.Conv1D(16, 4, activation='relu'))
    model.add(layers.GlobalMaxPooling1D())
    model.add(layers.Dense(1, activation='sigmoid')) #output 1 bc we only have 2 labels: attack or not attack
    return model
model = create_model_otids()

In [16]:
callbacks = [
    ModelCheckpoint("saved_models/best_model_OTIDS_cnn.keras", monitor='val_auc', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=4, min_lr=1e-12, verbose=1),
    EarlyStopping(monitor='val_auc', patience=12, verbose=1, restore_best_weights=True)
]
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",   # correct for sigmoid + 0/1 labels
    metrics=[
        "accuracy",
        keras.metrics.AUC(name="auc"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        F1Score(name="f1")
    ]
)
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=50,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50
957/957 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4948 - auc: 0.4947 - f1: 0.5123 - loss: 0.7029 - precision: 0.4966 - recall: 0.5333
Epoch 1: val_auc improved from inf to 0.49548, saving model to saved_models/best_model_OTIDS_cnn.keras
957/957 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.4948 - auc: 0.4947 - f1: 0.5123 - loss: 0.7029 - precision: 0.4966 - recall: 0.5333 - val_accuracy: 0.4924 - val_auc: 0.4955 - val_f1: 0.4211 - val_loss: 0.6963 - val_precision: 0.4968 - val_recall: 0.3653 - learning_rate: 0.0010
Epoch 2/50
954/957 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5090 - auc: 0.5156 - f1: 0.5425 - loss: 0.6940 - precision: 0.5146 - recall: 0.5768
Epoch 2: val_auc improved from 0.49548 to 0.49445, saving model to saved_models/best_model_OTIDS_cnn.keras
957/957 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.5090 - auc: 0.5156 - f1: 0.5424 - loss: 0.6940 - precision: 0.5146 - recall: 0.5766 - val_accuracy: 0.4971 - val_auc: 0.4944 - val_f1: 0.4178 - 

In [17]:
testing_acc = model.evaluate(X_test,y_test, verbose=1)
print(f"Test loss: {testing_acc[0]}")
print(f"Test accuracy: {testing_acc[1]}")
print(f"Test AUC: {testing_acc[2]}")
print(f"Test Precision: {testing_acc[3]}")
print(f"Test Recall: {testing_acc[4]}")
print(f"Test F1: {testing_acc[5]}")

266/266 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4980 - auc: 0.4983 - f1: 0.4294 - loss: 0.6956 - precision: 0.5004 - recall: 0.3764
Test loss: 0.6955057978630066
Test accuracy: 0.4983537197113037
Test AUC: 0.497335284948349
Test Precision: 0.5006142258644104
Test Recall: 0.3817330300807953
Test F1: 0.4331649839878082


In [8]:
#examine classification predictions
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)
print(classification_report(y_true, y_pred, target_names=["Normal", "DoS", "Fuzzy", "Impersonation"]))

#ROC AUC
#sensitivity (recall) and
# validity (precision)
from sklearn.metrics import roc_auc_score
roc_auc = roc_auc_score(y_test_cat, y_pred_probs, multi_class='ovr')
print(f"ROC AUC Score: {roc_auc}")

266/266 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


AxisError: axis 1 is out of bounds for array of dimension 1

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Normal", "DoS", "Fuzzy", "Impersonation"],
            yticklabels=["Normal", "DoS", "Fuzzy", "Impersonation"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()